In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded ✓")

Libraries loaded ✓


In [3]:
columns = [
    'checking_status', 'duration', 'credit_history', 'purpose',
    'credit_amount', 'savings_status', 'employment',
    'installment_commitment', 'personal_status', 'other_parties',
    'residence_since', 'property_magnitude', 'age',
    'other_payment_plans', 'housing', 'existing_credits',
    'job', 'num_dependents', 'own_telephone',
    'foreign_worker', 'class'
]

df = pd.read_csv('../data/original/german_credit_data.csv')

# Verification
print(f"Shape: {df.shape}")
print(f"\nClass distribution:\n{df['class'].value_counts()}")
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"\nDtypes:\n{df.dtypes}")

Shape: (1000, 21)

Class distribution:
class
good    700
bad     300
Name: count, dtype: int64

Missing values: 0

Dtypes:
checking_status             str
duration                  int64
credit_history              str
purpose                     str
credit_amount             int64
savings_status              str
employment                  str
installment_commitment    int64
personal_status             str
other_parties               str
residence_since           int64
property_magnitude          str
age                       int64
other_payment_plans         str
housing                     str
existing_credits          int64
job                         str
num_dependents            int64
own_telephone               str
foreign_worker              str
class                       str
dtype: object


In [4]:
drop_cols = [
    'residence_since',   # correlation = 0.00, p > 0.05
    'num_dependents',    # correlation = -0.00, p > 0.05
    'existing_credits',  # low variance, p > 0.05
    'own_telephone'      # 3% bivariate gap, p > 0.05
]

df = df.drop(columns=drop_cols)

print(f"Shape after dropping: {df.shape}")
print(f"Remaining columns:\n{df.columns.tolist()}")

Shape after dropping: (1000, 17)
Remaining columns:
['checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount', 'savings_status', 'employment', 'installment_commitment', 'personal_status', 'other_parties', 'property_magnitude', 'age', 'other_payment_plans', 'housing', 'job', 'foreign_worker', 'class']


In [5]:
job_map = {
    'skilled'                    : 'stable',
    'high qualif/self emp/mgmt'  : 'stable',
    'unskilled resident'         : 'unstable',
    'unemp/unskilled non res'    : 'unstable'
}

df['job'] = df['job'].map(job_map)

print("Job regrouped ✓")
print(f"\nValue counts:\n{df['job'].value_counts()}")
print(f"\nDefault rate by job group:")
print(df.groupby('job')['class'].apply(
    lambda x: (x == 'bad').sum() / len(x) * 100
).round(2))

Job regrouped ✓

Value counts:
job
stable      778
unstable    222
Name: count, dtype: int64

Default rate by job group:
job
stable      30.46
unstable    28.38
Name: class, dtype: float64


In [6]:
df = df.drop(columns=['job'])

print(f"Shape after dropping job: {df.shape}")
print(f"Remaining columns:\n{df.columns.tolist()}")

Shape after dropping job: (1000, 16)
Remaining columns:
['checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount', 'savings_status', 'employment', 'installment_commitment', 'personal_status', 'other_parties', 'property_magnitude', 'age', 'other_payment_plans', 'housing', 'foreign_worker', 'class']


## New Features

In [7]:
df['credit_amount_log'] = np.log1p(df['credit_amount'])

# Verify
print("credit_amount stats (original):")
print(df['credit_amount'].describe().round(2))

print("\ncredit_amount_log stats (transformed):")
print(df['credit_amount_log'].describe().round(2))

print(f"\nSkewness before: {df['credit_amount'].skew():.3f}")
print(f"Skewness after:  {df['credit_amount_log'].skew():.3f}")

credit_amount stats (original):
count     1000.00
mean      3271.26
std       2822.74
min        250.00
25%       1365.50
50%       2319.50
75%       3972.25
max      18424.00
Name: credit_amount, dtype: float64

credit_amount_log stats (transformed):
count    1000.00
mean        7.79
std         0.78
min         5.53
25%         7.22
50%         7.75
75%         8.29
max         9.82
Name: credit_amount_log, dtype: float64

Skewness before: 1.950
Skewness after:  0.130


In [8]:
df['monthly_burden'] = df['credit_amount'] / df['duration']

# Verify — check if it discriminates between good and bad
print("monthly_burden stats:")
print(df['monthly_burden'].describe().round(2))

print(f"\nMean monthly_burden by class:")
print(df.groupby('class')['monthly_burden'].mean().round(2))

print(f"\nMedian monthly_burden by class:")
print(df.groupby('class')['monthly_burden'].median().round(2))

monthly_burden stats:
count    1000.00
mean      167.69
std       153.49
min        24.06
25%        89.60
50%       130.33
75%       206.18
max      2482.67
Name: monthly_burden, dtype: float64

Mean monthly_burden by class:
class
bad     172.04
good    165.82
Name: monthly_burden, dtype: float64

Median monthly_burden by class:
class
bad     121.06
good    134.92
Name: monthly_burden, dtype: float64


Interesting and nuanced result. Let's read this carefully:

Mean: bad (172) > good (165) — bad borrowers have slightly higher mean burden
Median: good (134) > bad (121) — good borrowers have higher median burden

What this tells us:
The mean gap is only ~6 DM which is small. The median actually flips direction — suggesting bad borrowers tend to take longer loans with smaller amounts rather than large monthly burdens. The outliers (max 2482) are pulling the bad borrowers' mean up.
Decision: Keep it. It's a valid financial ratio and tree-based models like XGBoost can extract non-linear patterns from it even if the linear separation is weak. SHAP will tell us on Day 5 if it actually contributed.

In [9]:
checking_score_map = {
    '<0'          : 3,   # highest stress
    '0<=X<200'    : 2,
    '>=200'       : 1,
    'no checking' : 0    # lowest stress
}

savings_score_map = {
    '<100'            : 3,   # highest stress
    '100<=X<500'      : 2,
    '500<=X<1000'     : 1,
    '>=1000'          : 0,
    'no known savings': 0    # no savings account — ambiguous, treat as low stress
}

df['financial_stress_score'] = (
    df['checking_status'].map(checking_score_map) +
    df['savings_status'].map(savings_score_map)
)

print("financial_stress_score distribution:")
print(df['financial_stress_score'].value_counts().sort_index())

print(f"\nDefault rate by stress score:")
print(df.groupby('financial_stress_score')['class'].apply(
    lambda x: (x == 'bad').sum() / len(x) * 100
).round(2))

financial_stress_score distribution:
financial_stress_score
0    124
1     53
2    102
3    242
4     96
5    164
6    219
Name: count, dtype: int64

Default rate by stress score:
financial_stress_score
0     8.87
1    13.21
2    15.69
3    18.18
4    37.50
5    43.90
6    52.05
Name: class, dtype: float64


✅ Score 0 → 8.87% default rate
✅ Score 6 → 52.05% default rate
✅ Perfect monotonic increase — every single score level has a higher default rate than the one before it
✅ The gap between score 0 and score 6 is 43 percentage points

This is one of the strongest features we've engineered. It tells a clean, intuitive story — the more financially stressed a borrower is, the more likely they are to default. This will also make a great visual in  Streamlit app on Day 6.

In [10]:
df['high_risk_interaction'] = (
    (df['checking_status'].isin(['<0', '0<=X<200'])) &
    (df['credit_history'].isin(['all paid', 'no credits/all paid']))
).astype(int)

print("high_risk_interaction distribution:")
print(df['high_risk_interaction'].value_counts())

print(f"\nDefault rate by interaction flag:")
print(df.groupby('high_risk_interaction')['class'].apply(
    lambda x: (x == 'bad').sum() / len(x) * 100
).round(2))

high_risk_interaction distribution:
high_risk_interaction
0    932
1     68
Name: count, dtype: int64

Default rate by interaction flag:
high_risk_interaction
0    27.15
1    69.12
Name: class, dtype: float64


✅ Flag=0 → 27.15% default rate
✅ Flag=1 → 69.12% default rate
✅ A 42 percentage point gap from a single binary flag
✅ Validates our Day 1 discovery perfectly

Only 68 borrowers (6.8%) fall into this high-risk combination — but when they do, nearly 7 in 10 default. 

In [11]:
print(f"Shape: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")

Shape: (1000, 20)

Columns:
['checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount', 'savings_status', 'employment', 'installment_commitment', 'personal_status', 'other_parties', 'property_magnitude', 'age', 'other_payment_plans', 'housing', 'foreign_worker', 'class', 'credit_amount_log', 'monthly_burden', 'financial_stress_score', 'high_risk_interaction']


In [12]:
df = df.drop(columns=['credit_amount'])

print(f"Shape after dropping original credit_amount: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")

Shape after dropping original credit_amount: (1000, 19)

Columns:
['checking_status', 'duration', 'credit_history', 'purpose', 'savings_status', 'employment', 'installment_commitment', 'personal_status', 'other_parties', 'property_magnitude', 'age', 'other_payment_plans', 'housing', 'foreign_worker', 'class', 'credit_amount_log', 'monthly_burden', 'financial_stress_score', 'high_risk_interaction']


## Encoding

In [13]:
df['target'] = df['class'].map({'good': 0, 'bad': 1})
df = df.drop(columns=['class'])

print("Target encoded ✓")
print(f"\nTarget distribution:")
print(df['target'].value_counts())
print(f"\nShape: {df.shape}")

Target encoded ✓

Target distribution:
target
0    700
1    300
Name: count, dtype: int64

Shape: (1000, 19)


In [14]:
checking_order = {
    'no checking' : 0,
    '>=200'       : 1,
    '0<=X<200'    : 2,
    '<0'          : 3
}

savings_order = {
    '>=1000'          : 0,
    '500<=X<1000'     : 1,
    '100<=X<500'      : 2,
    '<100'            : 3,
    'no known savings': 4
}

employment_order = {
    '>=7'       : 0,
    '4<=X<7'    : 1,
    '1<=X<4'    : 2,
    '<1'        : 3,
    'unemployed': 4
}

df['checking_status']  = df['checking_status'].map(checking_order)
df['savings_status']   = df['savings_status'].map(savings_order)
df['employment']       = df['employment'].map(employment_order)

print("Ordinal encoding done ✓")
print(f"\nchecking_status values: {sorted(df['checking_status'].unique())}")
print(f"savings_status values:  {sorted(df['savings_status'].unique())}")
print(f"employment values:      {sorted(df['employment'].unique())}")

Ordinal encoding done ✓

checking_status values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
savings_status values:  [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
employment values:      [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


In [15]:
# Check what's left to encode
cat_remaining = df.select_dtypes(include='object').columns.tolist()
print("Still needs encoding:", cat_remaining)

Still needs encoding: ['credit_history', 'purpose', 'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 'housing', 'foreign_worker']


In [16]:
# Step 1 — Group purpose into risk buckets
high_risk_purpose = [
    'education', 'other', 'new car', 'repairs',
    'business', 'domestic appliance', 
    'furniture/equipment', 'radio/tv'
]

df['purpose_risk'] = df['purpose'].apply(
    lambda x: 'high_risk' if x in high_risk_purpose else 'low_risk'
)

print("Purpose risk distribution:")
print(df['purpose_risk'].value_counts())
print(f"\nDefault rate by purpose risk:")
print(df.groupby('purpose_risk')['target'].apply(
    lambda x: (x == 1).sum() / len(x) * 100
).round(2))

# Drop original purpose
df = df.drop(columns=['purpose'])

Purpose risk distribution:
purpose_risk
high_risk    888
low_risk     112
Name: count, dtype: int64

Default rate by purpose risk:
purpose_risk
high_risk    31.76
low_risk     16.07
Name: target, dtype: float64


In [17]:
# Step 2 — One-Hot Encode all remaining nominal categoricals
nominal_cols = [
    'credit_history', 'personal_status', 'other_parties',
    'property_magnitude', 'other_payment_plans', 
    'housing', 'foreign_worker', 'purpose_risk'
]

df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

print("One-hot encoding done ✓")
print(f"Shape after encoding: {df.shape}")
print(f"\nAll columns:\n{df.columns.tolist()}")

One-hot encoding done ✓
Shape after encoding: (1000, 29)

All columns:
['checking_status', 'duration', 'savings_status', 'employment', 'installment_commitment', 'age', 'credit_amount_log', 'monthly_burden', 'financial_stress_score', 'high_risk_interaction', 'target', 'credit_history_critical/other existing credit', 'credit_history_delayed previously', 'credit_history_existing paid', 'credit_history_no credits/all paid', 'personal_status_male div/sep', 'personal_status_male mar/wid', 'personal_status_male single', 'other_parties_guarantor', 'other_parties_none', 'property_magnitude_life insurance', 'property_magnitude_no known property', 'property_magnitude_real estate', 'other_payment_plans_none', 'other_payment_plans_stores', 'housing_own', 'housing_rent', 'foreign_worker_yes', 'purpose_risk_low_risk']


The gap is ~16 percentage points — meaningful but not dramatic.
More importantly — 888 vs 112 is a severe imbalance within purpose itself. Only 11.2% of borrowers fall into low_risk purpose. This means the feature may not contribute much signal — most borrowers are in high_risk regardless.
This is fine — XGBoost will figure out the weight. SHAP on Day 5 will tell us if it actually contributed.

In [19]:
# Save the fully engineered dataset
df.to_csv('../data/processed/german_credit_engineered.csv', index=False)
print(f"Engineered dataset saved ✓")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Engineered dataset saved ✓
Shape: (1000, 29)
Columns: ['checking_status', 'duration', 'savings_status', 'employment', 'installment_commitment', 'age', 'credit_amount_log', 'monthly_burden', 'financial_stress_score', 'high_risk_interaction', 'target', 'credit_history_critical/other existing credit', 'credit_history_delayed previously', 'credit_history_existing paid', 'credit_history_no credits/all paid', 'personal_status_male div/sep', 'personal_status_male mar/wid', 'personal_status_male single', 'other_parties_guarantor', 'other_parties_none', 'property_magnitude_life insurance', 'property_magnitude_no known property', 'property_magnitude_real estate', 'other_payment_plans_none', 'other_payment_plans_stores', 'housing_own', 'housing_rent', 'foreign_worker_yes', 'purpose_risk_low_risk']
